# Preprocessing

Before training the predictive models, a preprocessing pipeline was applied to prepare the dataset and improve data quality. This step included handling missing values, transforming categorical variables, feature engineering, and removing unnecessary attributes.

Given the clinical nature of the MIMIC-III dataset, different strategies were used to handle missing values depending on the type of variable. Continuous physiological and laboratory measurements were mainly imputed using the median, while intervention- and medication-related variables were filled with zero when appropriate, since missing values in these cases often indicate that a procedure or medication was not administered.

Categorical variables were transformed using one-hot encoding. For high-cardinality variables such as diagnosis, only the most frequent categories were preserved, while less frequent diagnoses were grouped into a common “OTHER” category to reduce dimensionality and sparsity.

Additionally, some features were engineered during preprocessing. For example, the date of birth was converted into patient age, which is more clinically relevant for predictive modeling. Identifier columns and non-informative attributes were removed before model training.

In [1]:
import pandas as pd
from src.feature_groups_map import feature_groups
from IPython.display import display

In [2]:
df = pd.read_csv("data/datasets/final_selected.csv")
diagnosis_map_df = pd.read_csv("data/datasets/diagnosis_map.csv")

In [3]:
df.shape

(62722, 159)

In [4]:
def get_missing_info(df):
    missing_info = pd.DataFrame({
        "null_count": df.isnull().sum(),
        "null_percentage": (df.isnull().sum() / len(df)) * 100
    })

    missing_info = missing_info.sort_values(
        by="null_percentage",
        ascending=False
    )

    display(
    missing_info.style.set_table_attributes(
        'style="display:inline-block; max-height:500px; overflow:auto;"'
    )
)

    return missing_info

In [5]:
missing_info = get_missing_info(df)

,null_count,null_percentage
dobutamine_count,62588,99.786359
dobutamine_sum,62588,99.786359
tpn_sum,62459,99.580689
tpn_count,62459,99.580689
cisatracurium_count,62436,99.544020
cisatracurium_sum,62436,99.544020
milrinone_count,62412,99.505756
milrinone_sum,62412,99.505756
cryoprecipitate_sum,62383,99.459520
dexmedetomidine_sum,62246,99.241096


Since the target variable of this project is Length of Stay (LOS), all rows with missing LOS values were removed from the dataset, as these samples cannot be used for supervised learning. Additionally, rows with missing diagnosis information were also excluded. As only a small number of records (approximately 25) lacked diagnosis data, removing them had minimal impact on the overall dataset while helping maintain data consistency during preprocessing.

In [6]:
df = df[df["LOS"].notnull()]
df = df[df["DIAGNOSIS"].notnull()]

To handle missing values, different imputation strategies were applied according to the semantic meaning of each feature group. Variables related to medications, interventions, procedures, outputs, and device usage were imputed with zero, since missing values in these cases often indicate that the event or intervention did not occur during the patient stay. On the other hand, continuous physiological and laboratory measurements were imputed using the median value of each feature. Median imputation was chosen because it is more robust to outliers and skewed distributions, which are common in clinical datasets such as MIMIC-III.

In [7]:
for col in feature_groups["zero_impute"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [8]:
for col in feature_groups["median_impute"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [9]:
missing_info = get_missing_info(df)

,null_count,null_percentage
gcs_verbal_avg,0,0.000000
gcs_eye_avg,0,0.000000
gcs_verbal_min,0,0.000000
systolic_bp_min,0,0.000000
respiratory_rate_min,0,0.000000
ph_blood_latest,0,0.000000
gcs_eye_min,0,0.000000
gcs_total_avg,0,0.000000
gcs_motor_avg,0,0.000000
gcs_verbal_max,0,0.000000


In [10]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns

print(categorical_columns)

Index(['ADMISSION_TYPE', 'DIAGNOSIS', 'DOB', 'ADMITTIME'], dtype='str')


C:\Users\belac\AppData\Local\Temp\ipykernel_34224\2331803844.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


After handling missing values, additional preprocessing steps were applied to prepare the categorical and temporal information for modeling. Since raw date variables are not directly suitable for machine learning algorithms, the patient’s date of birth was transformed into an age feature by calculating the difference between admission time and birth date. This approach provides a more clinically meaningful representation of patient demographics. After generating the age variable, the original DOB and ADMITTIME columns were removed from the dataset to avoid redundancy and reduce unnecessary temporal information during training.

In [11]:
df = df.copy()

df = df.assign(
    age=((pd.to_datetime(df["ADMITTIME"]) -
          pd.to_datetime(df["DOB"])).dt.days / 365.25).astype(int)
)

columns_to_drop = [
    "DOB",
    "ADMITTIME"
]

df = df.drop(columns=columns_to_drop)

In [12]:
len(df["DIAGNOSIS"].value_counts())

15248

The DIAGNOSIS column originally contained 15,248 unique categories, making it impractical to directly apply one-hot encoding or use the raw values for model training due to the extremely high dimensionality and sparsity that would be introduced into the dataset. To address this issue, a custom script named src.create_diagnosis_map was developed to automatically group diagnoses into a smaller set of clinically meaningful categories using a Large Language Model (LLM).

The diagnoses were classified into predefined medical groups, including categories such as cardiovascular, respiratory, infectious, neurological, gastrointestinal, renal, oncology, trauma-related conditions, among others. To improve processing efficiency, the classification pipeline was executed in parallel using 10 workers (MAX_WORKERS = 10), with batches of 50 diagnoses sent per API request (BATCH_SIZE = 50). Additionally, intermediate results were periodically saved every 10 completed batches (SAVE_EVERY_BATCHES = 10) to avoid losing progress during long-running executions.

After generating the diagnosis-category mapping, the original diagnosis values were replaced by their corresponding grouped category using a dictionary-based mapping approach. This significantly reduced the cardinality of the feature while preserving clinically relevant information for the predictive modeling task.

In [13]:
diagnosis_dict = dict(
    zip(
        diagnosis_map_df["diagnosis"],
        diagnosis_map_df["category"]
    )
)

df["DIAGNOSIS"] = df["DIAGNOSIS"].map(diagnosis_dict)

In [14]:
len(df["DIAGNOSIS"].value_counts())

20

In [15]:
df["DIAGNOSIS"].value_counts()

DIAGNOSIS
cardiovascular            14942
other                      8425
neurological               6159
infectious                 5588
gastrointestinal           4819
symptoms_unspecified       4480
trauma_injury              3745
respiratory                2778
oncology                   2059
hepatobiliary              1472
endocrine_metabolic        1445
renal                      1105
hematologic                 808
toxicology_poisoning        617
genitourinary               479
musculoskeletal             459
postoperative_surgical      353
psychiatric                 311
pregnancy_obstetric         238
dermatologic                 52
Name: count, dtype: int64

After that, one-hot encoding was applied to transform categorical variables into a numerical representation suitable for machine learning models. This technique converts each categorical value into a binary feature, allowing the algorithms to interpret categorical information without introducing artificial ordinal relationships between categories.

In [16]:
df = pd.get_dummies(
    df,
    columns=["DIAGNOSIS", "ADMISSION_TYPE"],
    drop_first=True,
    dtype=int
)

In [17]:
df.head().shape

(5, 178)

In [18]:
df.head()

,gcs_verbal_avg,gcs_eye_avg,gcs_verbal_min,systolic_bp_min,respiratory_rate_min,ph_blood_latest,gcs_eye_min,gcs_total_avg,gcs_motor_avg,gcs_verbal_max,...,DIAGNOSIS_pregnancy_obstetric,DIAGNOSIS_psychiatric,DIAGNOSIS_renal,DIAGNOSIS_respiratory,DIAGNOSIS_symptoms_unspecified,DIAGNOSIS_toxicology_poisoning,DIAGNOSIS_trauma_injury,ADMISSION_TYPE_EMERGENCY,ADMISSION_TYPE_NEWBORN,ADMISSION_TYPE_URGENT
0,5.00,3.833333,5.0,89.0,15.0,7.40,3.0,13.8,6.00,5.0,...,0,0,0,0,0,0,0,1,0,0
1,1.80,1.800000,1.0,86.0,0.0,7.41,1.0,13.8,5.15,5.0,...,0,0,0,0,1,0,0,1,0,0
2,5.00,4.000000,5.0,112.0,11.0,4.30,4.0,15.0,6.00,5.0,...,0,0,0,0,0,0,1,1,0,0
3,4.75,4.000000,4.0,95.0,11.0,118.00,4.0,13.8,6.00,5.0,...,0,0,0,0,0,0,0,1,0,0
4,5.00,4.000000,5.0,120.0,16.0,16.00,4.0,15.0,6.00,5.0,...,0,0,0,0,0,0,0,1,0,0


In [19]:
df.to_csv("data/datasets/final_processed.csv")